# NB14 — RQ2 Final Train and Evaluate

Compares frozen RQ2 arms **D0**, **D-Random**, and **D-Quality** with the RQ1 Direct protocol.

- **D0**: the already-frozen RQ1 Direct checkpoint. NB14 binds it and does not retrain it.
- **D-Random** and **D-Quality** CONTINUE FROM that same frozen D0 model state (learned weights, not public XLS-R+mBART reinitialization), using the same supervised G_train plus the frozen NB13 selection of that arm.
- Gold:pseudo mixing is an explicit configured slot policy, identical for both treatments and independent of unique pseudo row count. It stays `UNSET_REQUIRE_EXPLICIT_CONFIG` until a human sets it.
- Seed policy is explicit (multi-seed or declared compute-constrained single-seed). It is not hardcoded.
- Validation is frozen **G_validation** only. **G_test** stays closed until independent D-Random/D-Quality proofs succeed and a human unlocks it.
- Metrics are RQ1 SacreBLEU + chrF++ with paired cluster bootstrap on `group_id`.

Default flags start no training and do not open G_test. A normal Run All is safe.


## 1. Imports and runtime versions

In [ ]:
import json
import sys
from pathlib import Path

def _find_root(start: Path) -> Path:
    for cand in [start, *start.parents]:
        if (cand / "requirements.txt").is_file() and (cand / "src").is_dir() and (cand / "notebooks").is_dir():
            return cand
    raise RuntimeError("cannot locate project root")

PROJECT_ROOT = _find_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.rq2_final_contract import (
    ARM_D0, ARM_QUALITY, ARM_RANDOM, ARMS, FINAL_RELATIVE_DIR, Nb14Flags,
    UNLOCK_REQUIRED, assert_augmentation_fairness, assert_nb14_output_dir,
    build_arm_training_contract, compute_nb14_source_fingerprint, default_nb14_flags,
    evaluation_protocol, experiment_root_for_arm, locked_architecture, require_configured_mix_policy,
    require_configured_seed_policy, resolve_d0_best_checkpoint_dir, resolve_frozen_d0_init, verify_upstream_rq2,
)
from src.rq2_final_data import (
    compose_arm_training_rows, data_contract_payload, load_frozen_supervised_splits,
    load_pinned_nb13_arm_manifest, validation_identity, verify_equal_budget_from_nb13,
    verify_pinned_nb13_selection,
    write_arm_manifest_csv,
)
from src.rq2_final_train import (
    bind_d0_arm, build_hparams_from_yaml, ensure_arm_root, experiment_fingerprint,
    freeze_best_checkpoint, merge_training_fields, resume_or_train_arm, select_best_checkpoint,
    training_rows_to_frame, validation_history_from_checkpoints,
)
from src.rq2_final_evaluate import (
    arm_metrics, assert_paired_alignment, derive_final_readiness, generate_from_direct_checkpoint,
    load_frozen_test_frame, paired_cluster_bootstrap_arms, persist_final_artifacts,
    publish_final_generation, require_g_test_unlocked, run_g_test_inference,
)
from src.direct_runtime_paths import resolve_direct_runtime_paths

print("PROJECT_ROOT:", PROJECT_ROOT)
print("architecture:", json.dumps(locked_architecture(), indent=2, sort_keys=True)[:400])
print("source_fingerprint:", compute_nb14_source_fingerprint()["aggregate_sha256"])


## 2. Safe flags

`RUN_REAL_TRAINING`, `ALLOW_G_TEST_EVALUATION`, and `RQ2_FINAL_FROZEN` stay False until
NB11/NB12/NB13 are frozen and this notebook has been reviewed. Fill `DIRECT_STATE_DIR`
with the RQ1 Direct durable state directory before a real run.

`GOLD_PSEUDO_MIX_POLICY` and `SEED_POLICY` stay `UNSET_REQUIRE_EXPLICIT_CONFIG` until a
human configures them. Real training refuses to start while they are unset. Do not invent
a scientific gold:pseudo ratio or seed set here.


In [ ]:
RUN_REAL_TRAINING = False
ALLOW_G_TEST_EVALUATION = False
RQ2_FINAL_FROZEN = False
DIRECT_STATE_DIR = ""
GOLD_PSEUDO_MIX_POLICY = "UNSET_REQUIRE_EXPLICIT_CONFIG"
GOLD_SLOTS = None
PSEUDO_SLOTS = None
SEED_POLICY = "UNSET_REQUIRE_EXPLICIT_CONFIG"
SEED_POLICY_SEEDS = []

FLAGS = Nb14Flags(
    run_real_training=RUN_REAL_TRAINING,
    allow_g_test_evaluation=ALLOW_G_TEST_EVALUATION,
    rq2_final_frozen=RQ2_FINAL_FROZEN,
    direct_state_dir=DIRECT_STATE_DIR,
    durable_root="",
)
print(json.dumps(FLAGS.as_dict(), indent=2, sort_keys=True))
print("GOLD_PSEUDO_MIX_POLICY:", GOLD_PSEUDO_MIX_POLICY, GOLD_SLOTS, PSEUDO_SLOTS)
print("SEED_POLICY:", SEED_POLICY, SEED_POLICY_SEEDS)
if RUN_REAL_TRAINING and not RQ2_FINAL_FROZEN:
    raise RuntimeError("RUN_REAL_TRAINING requires RQ2_FINAL_FROZEN=True after review")
if ALLOW_G_TEST_EVALUATION and not RQ2_FINAL_FROZEN:
    raise RuntimeError("ALLOW_G_TEST_EVALUATION requires RQ2_FINAL_FROZEN=True")
if RUN_REAL_TRAINING:
    require_configured_mix_policy(
        {"gold_pseudo_mix_policy": GOLD_PSEUDO_MIX_POLICY, "gold_slots": GOLD_SLOTS, "pseudo_slots": PSEUDO_SLOTS},
        for_real_training=True,
    )
    require_configured_seed_policy(
        {"seed_policy": SEED_POLICY, "seed_policy_seeds": SEED_POLICY_SEEDS},
        for_real_training=True,
    )


## 3. Resolve runtime paths

In [ ]:
OUT_DIR = PROJECT_ROOT / FINAL_RELATIVE_DIR
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR = assert_nb14_output_dir(OUT_DIR, PROJECT_ROOT)
PATHS = resolve_direct_runtime_paths(project_root=PROJECT_ROOT)
print("NB14 output:", OUT_DIR.relative_to(PROJECT_ROOT))
print("local_root:", PATHS.local_root)
print("durable_root:", PATHS.durable_root)
FLAGS = Nb14Flags(
    run_real_training=FLAGS.run_real_training,
    allow_g_test_evaluation=FLAGS.allow_g_test_evaluation,
    rq2_final_frozen=FLAGS.rq2_final_frozen,
    direct_state_dir=FLAGS.direct_state_dir or str(getattr(PATHS, 'direct_state_dir', '') or ''),
    durable_root=str(PATHS.durable_root),
)
print(json.dumps(FLAGS.as_dict(), indent=2, sort_keys=True))


## 4. Load and verify upstream contracts

In [ ]:
UPSTREAM = None
if not DIRECT_STATE_DIR:
    print("upstream verify: skipped (DIRECT_STATE_DIR empty; default Run All is safe)")
else:
    UPSTREAM = verify_upstream_rq2(PROJECT_ROOT, flags=FLAGS)
    print("nb11:", UPSTREAM["nb11_generation_id"])
    print("nb12:", UPSTREAM["nb12_generation_id"])
    print("nb13:", UPSTREAM["nb13_generation_id"])
    print("d0_policy:", UPSTREAM["d0"]["d0_policy"])
    print("d0_checkpoint_fingerprint:", UPSTREAM["d0"]["checkpoint_fingerprint_sha256"])

## 5. Build NB14 data contracts

In [ ]:
DATA = None
SUPERVISED = None
if UPSTREAM is None:
    print("data contracts: not built")
else:
    SUPERVISED = load_frozen_supervised_splits(
        PROJECT_ROOT,
        d0_identity=UPSTREAM["d0"],
        durable_root=FLAGS.durable_root or None,
    )
    g_train = SUPERVISED["train"]
    g_val = SUPERVISED["validation"]
    NB13_SELECTION = verify_pinned_nb13_selection(PROJECT_ROOT, generation_id=UPSTREAM["nb13_generation_id"])
    budget = verify_equal_budget_from_nb13(NB13_SELECTION)
    val_id = validation_identity(g_val)
    DATA = {}
    for arm in (ARM_D0, ARM_RANDOM, ARM_QUALITY):
        pseudo = [] if arm == ARM_D0 else load_pinned_nb13_arm_manifest(
            PROJECT_ROOT, arm=arm, upstream=UPSTREAM,
        )
        composition = compose_arm_training_rows(
            arm=arm,
            supervised_rows=g_train.to_dict("records"),
            pseudo_rows=pseudo,
            selection_contract_sha256=UPSTREAM["nb13_selection_contract_sha256"],
            nb12_contract_sha256=UPSTREAM["nb12_contract_sha256"],
        )
        manifest_path = OUT_DIR / f"{arm}_train_manifest.csv"
        sha = write_arm_manifest_csv(manifest_path, composition["rows"])
        DATA[arm] = {
            "composition": composition,
            "data_contract": data_contract_payload(
                composition, manifest_sha256=sha, validation=val_id, upstream=UPSTREAM, budget=budget,
            ),
            "manifest_sha256": sha,
            "train_frame": training_rows_to_frame(composition["rows"]),
            "validation_frame": g_val,
        }
        print(arm, "rows", composition["n_rows"], "pseudo", composition["pseudo"]["n_rows"])


## 6. Build per-arm training contracts

In [ ]:
CONTRACTS = None
LAYOUTS = None
D0_INIT = None
if UPSTREAM is None or DATA is None:
    print("training contracts: not built")
else:
    from src.rq2_final_contract import assert_augmentation_fairness, compute_nb14_source_fingerprint
    FAIRNESS_PROOF = None
    hparams = build_hparams_from_yaml(PROJECT_ROOT)
    source_fp = compute_nb14_source_fingerprint()["aggregate_sha256"]
    D0_INIT = resolve_frozen_d0_init(FLAGS, project_root=PROJECT_ROOT, identity=UPSTREAM["d0"])
    mix_policy = {
        "gold_pseudo_mix_policy": GOLD_PSEUDO_MIX_POLICY,
        "gold_slots": GOLD_SLOTS,
        "pseudo_slots": PSEUDO_SLOTS,
    }
    seed_policy = {
        "seed_policy": SEED_POLICY,
        "seed_policy_seeds": list(SEED_POLICY_SEEDS),
    }
    CONTRACTS = {}
    LAYOUTS = {}
    for arm in (ARM_D0, ARM_RANDOM, ARM_QUALITY):
        fields = merge_training_fields(
            arm=arm,
            data_contract=DATA[arm]["data_contract"],
            hparams=hparams,
            source_fingerprint_sha256=source_fp,
            rq1_direct_training_contract_hash=UPSTREAM["d0"]["direct_training_contract_hash"],
            d0_init=D0_INIT,
            mix_policy=mix_policy,
            seed_policy=seed_policy,
        )
        if arm != ARM_D0 and mix_policy["gold_pseudo_mix_policy"] == "UNSET_REQUIRE_EXPLICIT_CONFIG":
            print(arm, "training contract not frozen: GOLD_PSEUDO_MIX_POLICY is unset")
            CONTRACTS = None
            break
        contract = build_arm_training_contract(fields)
        fingerprint = experiment_fingerprint(
            arm=arm,
            training_contract=contract,
            data_manifest_sha256=DATA[arm]["manifest_sha256"],
            model_revision=contract["decoder_revision"],
        )
        root = experiment_root_for_arm(
            durable_root=PATHS.durable_root,
            contract_hash=contract["arm_training_contract_sha256"],
            arm=arm,
        )
        CONTRACTS[arm] = {"contract": contract, "fingerprint": fingerprint}
        LAYOUTS[arm] = ensure_arm_root(root)
        print(arm, "contract", contract["arm_training_contract_sha256"][:16], "init", contract.get("init_policy"), "root", root.name)
    if CONTRACTS is not None:
        assert_augmentation_fairness(CONTRACTS[ARM_RANDOM]["contract"], CONTRACTS[ARM_QUALITY]["contract"])
        FAIRNESS_PROOF = assert_augmentation_fairness(CONTRACTS[ARM_RANDOM]["contract"], CONTRACTS[ARM_QUALITY]["contract"])
        print("fairness_proof", FAIRNESS_PROOF["fairness_proof_sha256"])
        print("d0_init_model_state", D0_INIT["model_state_sha256"])


## 7. Preflight report

In [ ]:
print("RUN_REAL_TRAINING:", RUN_REAL_TRAINING)
print("ALLOW_G_TEST_EVALUATION:", ALLOW_G_TEST_EVALUATION)
print("RQ2_FINAL_FROZEN:", RQ2_FINAL_FROZEN)
print("UNLOCK_REQUIRED:", UNLOCK_REQUIRED)

## 8. Optional REAL training gate

In [ ]:
if not RUN_REAL_TRAINING:
    print("real training gate: closed")
else:
    if UPSTREAM is None:
        raise RuntimeError("refusing training: upstream was not verified")

## 9. Train or resume arms

In [ ]:
TRAIN_RESULTS = {}
if CONTRACTS is None or LAYOUTS is None or DATA is None:
    print("train/resume: skipped (contracts not built; default Run All is safe)")
else:
    for arm in (ARM_D0, ARM_RANDOM, ARM_QUALITY):
        TRAIN_RESULTS[arm] = resume_or_train_arm(
            arm=arm,
            flags=FLAGS,
            layout=LAYOUTS[arm],
            training_contract=CONTRACTS[arm]["contract"],
            fingerprint=CONTRACTS[arm]["fingerprint"],
            train_frame=DATA[arm]["train_frame"],
            validation_frame=DATA[arm]["validation_frame"],
            audio_cache_roots=[PATHS.audio_cache_dir],
        )
        print(arm, TRAIN_RESULTS[arm].get("reason") or TRAIN_RESULTS[arm].get("trained"))
    if not RUN_REAL_TRAINING:
        print("real HuggingFace Trainer.train() was not called")

## 10. Freeze best checkpoints from validation

In [ ]:
BEST = {}
if not LAYOUTS or not CONTRACTS:
    print("best-checkpoint freeze: skipped")
else:
    for arm in ARMS:
        if arm == ARM_D0:
            BEST[arm] = json.loads(LAYOUTS[arm]["best_checkpoint"].read_text(encoding="utf-8")) if LAYOUTS[arm]["best_checkpoint"].is_file() else {}
            continue
        history = validation_history_from_checkpoints(
            LAYOUTS[arm],
            arm=arm,
            training_contract_sha256=CONTRACTS[arm]["contract"]["arm_training_contract_sha256"],
            data_contract_sha256=CONTRACTS[arm]["contract"]["data_manifest_sha256"],
        )
        if not history:
            print(arm, "no validation history yet")
            continue
        selected = select_best_checkpoint(history)
        BEST[arm] = freeze_best_checkpoint(
            LAYOUTS[arm],
            selected,
            arm=arm,
            contract_hash=CONTRACTS[arm]["contract"]["arm_training_contract_sha256"],
            data_hash=CONTRACTS[arm]["contract"]["data_manifest_sha256"],
        )
        print(arm, BEST[arm]["checkpoint_name"], BEST[arm]["validation_metric"])


## 11. Final evaluation readiness report

In [ ]:
PROTOCOL = evaluation_protocol(UPSTREAM["rq1_eval"]) if UPSTREAM else None
READINESS = derive_final_readiness(
    flags=FLAGS,
    upstream=UPSTREAM,
    layouts=LAYOUTS,
    contracts=CONTRACTS,
    best=BEST if "BEST" in dir() else None,
    protocol=PROTOCOL,
)
print(json.dumps({"flat": READINESS["flat"], "all_ok": READINESS["all_ok"]}, indent=2, sort_keys=True))

## 12. Explicit G_test unlock gate

In [ ]:
if ALLOW_G_TEST_EVALUATION:
    require_g_test_unlocked(
        FLAGS,
        READINESS,
        layouts=LAYOUTS,
        contracts=CONTRACTS,
        best=BEST,
        d0_identity=UPSTREAM["d0"] if UPSTREAM else None,
        upstream=UPSTREAM,
    )
    print("G_test unlock: accepted")
else:
    print("G_test unlock: blocked (ALLOW_G_TEST_EVALUATION=False)")


## 13. G_test inference

In [ ]:
FRAMES = None
TEST_FRAME = None
if not ALLOW_G_TEST_EVALUATION:
    print("G_test inference: skipped")
else:
    require_g_test_unlocked(
        FLAGS, READINESS, layouts=LAYOUTS, contracts=CONTRACTS, best=BEST,
        d0_identity=UPSTREAM["d0"], upstream=UPSTREAM,
    )
    TEST_FRAME = load_frozen_test_frame(
        project_root=PROJECT_ROOT, flags=FLAGS, readiness=READINESS,
        layouts=LAYOUTS, contracts=CONTRACTS, best=BEST,
        d0_identity=UPSTREAM["d0"], upstream=UPSTREAM,
    )
    FRAMES = {}
    for arm in ARMS:
        payload = BEST.get(arm) or {}
        fingerprint = str(payload.get("checkpoint_fingerprint") or payload.get("checkpoint_fingerprint_sha256") or "")
        ckpt_name = str(payload.get("checkpoint_name") or "")
        ckpt_dir = resolve_d0_best_checkpoint_dir(FLAGS.direct_state_dir, UPSTREAM["d0"]) if arm == ARM_D0 else LAYOUTS[arm]["checkpoints"] / ckpt_name
        generate_fn = generate_from_direct_checkpoint(
            checkpoint_dir=ckpt_dir,
            training_contract=CONTRACTS[arm]["contract"],
            audio_cache_roots=[PATHS.audio_cache_dir],
        )
        FRAMES[arm] = run_g_test_inference(
            arm=arm,
            test_frame=TEST_FRAME,
            checkpoint_fingerprint=fingerprint,
            generate_fn=generate_fn,
        )
        print(arm, "predictions", len(FRAMES[arm]))


## 14. Metrics

In [ ]:
METRICS = None
if FRAMES is None:
    print("metrics: skipped")
else:
    uids = assert_paired_alignment(FRAMES)
    print("paired rows", len(uids))
    METRICS = {arm: arm_metrics(FRAMES[arm]) for arm in ARMS}
    print(json.dumps(METRICS, indent=2, sort_keys=True, default=str))

## 15. Paired cluster bootstrap

In [ ]:
BOOTSTRAP = None
if FRAMES is None or PROTOCOL is None:
    print("bootstrap: skipped")
else:
    BOOTSTRAP = paired_cluster_bootstrap_arms(
        FRAMES,
        n_samples=int(PROTOCOL["bootstrap_samples"]),
        seed=int(PROTOCOL["seed"]),
        confidence=float(PROTOCOL["confidence"]),
    )
    print("n_clusters", BOOTSTRAP["n_clusters"], "n_samples", BOOTSTRAP["n_samples"])

## 16. Final contract and artifact verification

In [ ]:
ARTIFACTS = None
if FRAMES is None or BOOTSTRAP is None or UPSTREAM is None:
    print("final contract: skipped (G_test not evaluated)")
else:
    from src.rq1_contract import ordered_uid_hash, reference_content_hash, sha256_file
    from src.rq2_final_contract import build_final_contract
    test_path = PROJECT_ROOT / "data" / "manifests" / "rq1_test.csv"
    fields = {
        "nb11_generation_id": UPSTREAM["nb11_generation_id"],
        "nb11_input_contract_sha256": UPSTREAM["nb11_input_contract_sha256"],
        "nb12_generation_id": UPSTREAM["nb12_generation_id"],
        "nb12_contract_sha256": UPSTREAM["nb12_contract_sha256"],
        "nb13_selection_contract_sha256": UPSTREAM["nb13_selection_contract_sha256"],
        "nb13_generation_id": UPSTREAM["nb13_generation_id"],
        "d_random_manifest_sha256": UPSTREAM["d_random_manifest_sha256"],
        "d_quality_manifest_sha256": UPSTREAM["d_quality_manifest_sha256"],
        "d0_direct_training_contract_hash": UPSTREAM["d0"]["direct_training_contract_hash"],
        "d0_checkpoint_fingerprint": BEST[ARM_D0]["checkpoint_fingerprint"],
        "d_random_training_contract_sha256": CONTRACTS[ARM_RANDOM]["contract"]["arm_training_contract_sha256"],
        "d_quality_training_contract_sha256": CONTRACTS[ARM_QUALITY]["contract"]["arm_training_contract_sha256"],
        "d_random_best_checkpoint": BEST[ARM_RANDOM]["checkpoint_name"],
        "d_quality_best_checkpoint": BEST[ARM_QUALITY]["checkpoint_name"],
        "g_test_manifest_sha256": sha256_file(test_path),
        "g_test_ordered_uid_hash": ordered_uid_hash(TEST_FRAME),
        "g_test_reference_hash": reference_content_hash(TEST_FRAME),
        "source_fingerprint_sha256": compute_nb14_source_fingerprint()["aggregate_sha256"],
        "bootstrap_config": PROTOCOL,
        "fairness_proof": FAIRNESS_PROOF,
        "d_random_training_complete": json.loads(LAYOUTS[ARM_RANDOM]["training_complete"].read_text(encoding="utf-8")),
        "d_quality_training_complete": json.loads(LAYOUTS[ARM_QUALITY]["training_complete"].read_text(encoding="utf-8")),
    }
    ARTIFACTS = persist_final_artifacts(
        OUT_DIR / "_stage_artifacts",
        frames=FRAMES,
        contract_fields=fields,
        best=BEST,
        n_samples=int(PROTOCOL["bootstrap_samples"]),
        seed=int(PROTOCOL["seed"]),
        confidence=float(PROTOCOL["confidence"]),
    )
    print("staged artifacts", sorted(ARTIFACTS))


## 17. Publish CURRENT only on success

In [ ]:
if ARTIFACTS is None or not ALLOW_G_TEST_EVALUATION or not RQ2_FINAL_FROZEN:
    print("publication: skipped unless every success gate is true")
else:
    published = publish_final_generation(PROJECT_ROOT, artifacts=ARTIFACTS, durable_root=str(PATHS.durable_root))
    print(json.dumps({"status": published["status"], "wrote_current": published["wrote_current"]}, indent=2))
    if published["status"] != "SUCCESS_RQ2_FINAL":
        raise RuntimeError(published.get("error") or "NB14 publication failed independent verification")
print("RUN_REAL_TRAINING:", RUN_REAL_TRAINING)
print("ALLOW_G_TEST_EVALUATION:", ALLOW_G_TEST_EVALUATION)
print("RQ2_FINAL_FROZEN:", RQ2_FINAL_FROZEN)
